# 11 Serialization

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Understand where Spark serializes data and code (shuffles, caching, Python ↔ JVM, closures shipped to executors), why it costs time, the difference between Java, Kryo, Tungsten's binary format, pickle and Arrow, and how to avoid the common serialization errors and slowdowns.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Serialization is a hidden cost in shuffles, UDFs and <code>toPandas()</code>, and a common source of confusing errors such as <code>PicklingError: Could not serialize object</code>. Interviewers ask "Java vs Kryo serialization?" and "why is a Python UDF slow?". Both answers are about serialization.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Where Spark serializes</b><br>
| Moment | What is serialized | Format |<br>|---|---|---|<br>| Shuffle and caching of DataFrames | Rows | <b>Tungsten binary format</b> (UnsafeRow), compact and schema-aware |<br>| Shuffle and caching of RDDs / JVM objects | Objects | Java serialization (default) or <b>Kryo</b> (faster, smaller) |<br>| Tasks sent to executors | Your functions (closures) and the variables they capture | Java serialization (JVM), <b>cloudpickle</b> (Python) |<br>| Python UDFs | Each row's values, both ways | <b>pickle</b> (classic UDFs) or <b>Apache Arrow</b> (pandas and Arrow UDFs) |<br>| <code>toPandas()</code>, <code>createDataFrame(pandas_df)</code> | Whole columns | <b>Arrow</b> (when enabled, and always with Spark Connect) |<br>| Spark Connect results | Result rows to the notebook | Arrow batches over gRPC |
</div>

```
DataFrame operations                    Python UDF
JVM rows (UnsafeRow) ──▶ JVM            JVM rows ──serialize (pickle/Arrow)──▶ Python worker
no conversion at all                              ◀──serialize back──────────── results
```

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A data scientist wrote a UDF that looked up product names in a Python dictionary of 2 million entries defined in the notebook. Every task shipped that dictionary to the executors as part of the UDF's closure, and the job spent most of its time serializing. Turning the dictionary into a DataFrame and <b>joining</b> (broadcast) made the job 10 times faster and removed a "task size is too large" warning.
</div>

## 1. DataFrames vs RDDs: why the format matters

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Explains why DataFrame shuffles are cheaper than RDD shuffles of Python or Java objects.<br><br>
<b>Why it matters</b><br>DataFrame rows are stored in Tungsten's compact binary layout, which Spark knows from the schema. They're shuffled and cached without generic object serialization, and compared and hashed without deserializing. RDDs of objects have to be serialized with Java serialization or Kryo, which is slower and larger. That's one of the reasons DataFrames outperform RDDs.
</div>

| Data | Shuffle / cache serialization | Notes |
|---|---|---|
| DataFrame / Dataset rows | Tungsten UnsafeRow (binary) | Schema known, no per-object metadata |
| RDD of JVM objects | Java (default) or Kryo | Kryo: faster and smaller, register classes for best results |
| RDD of Python objects | Pickled bytes | PySpark RDDs, not available on serverless |

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
The serializer (<code>spark.serializer</code>, for example <code>org.apache.spark.serializer.KryoSerializer</code>) is a cluster-level setting fixed at startup. It can't be changed from a notebook, and it isn't configurable on serverless. Kryo matters mainly for RDD-based code.
</div>

## 2. Code is serialized too: closures

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows that a UDF's body <b>and every variable it captures</b> is pickled and shipped to the executors with each task.<br><br>
<b>Why it matters</b><br>Anything a function references from the notebook (a dictionary, a model, a config object) travels with the task. Small values are fine. Big objects make every task heavy, and some objects can't be serialized at all.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The UDF uses the captured <code>multiplier</code> value correctly on the executors: <code>id * 3</code>.
</div>

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import udf

multiplier = 3                      # defined on the driver

@udf("long")
def times_multiplier(x):
    return x * multiplier           # captured in the closure and shipped to executors

spark.range(5).select("id", times_multiplier("id").alias("result")).show()

## 3. A classic error: capturing something that can't be serialized

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Captures a thread lock in a UDF's closure and shows the resulting <code>PicklingError</code>.<br><br>
<b>Why it matters</b><br>Locks, open files, database connections, the SparkSession itself and many client objects can't be pickled. The error appears when the job is submitted. The fix is to create such objects <b>inside</b> the function (or once per partition with <code>mapInPandas</code>), not to capture them from the notebook.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>PicklingError</code> (or a similar serialization error) mentioning that a <code>_thread.lock</code> object cannot be pickled.
</div>

In [0]:
import threading

lock = threading.Lock()

@udf("long")
def uses_lock(x):
    with lock:                       # the lock lives on the driver and can't be shipped
        return x

try:
    spark.range(3).select(uses_lock("id")).collect()
except Exception as e:
    print(type(e).__name__, "-", str(e).splitlines()[0][:160])

## 4. Big captured objects: closure vs join

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Looks up a name for 200,000 IDs in a 500,000-entry dictionary, first by capturing the dictionary in a UDF and then by turning it into a DataFrame and joining.<br><br>
<b>Why it matters</b><br>The UDF version pickles the whole dictionary into the tasks, then pays per-row Python serialization on top. The join version keeps everything in the JVM, and Spark can broadcast the lookup table efficiently. With Spark Connect (serverless), the old <code>sparkContext.broadcast()</code> variable isn't available, so a DataFrame join is the standard pattern for lookups.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Both give the same names. The join version is several times faster.
</div>

In [0]:
import time

lookup = {i: f"name_{i}" for i in range(500_000)}
ids = spark.range(0, 200_000, 1, numPartitions=8)

@udf("string")
def name_of(x):
    return lookup.get(x)            # the 500k-entry dict is pickled with the function

start = time.time()
via_udf = ids.select("id", name_of("id").alias("name"))
via_udf.write.format("noop").mode("overwrite").save()
print(f"closure UDF : {time.time() - start:.2f}s")

lookup_df = spark.createDataFrame(list(lookup.items()), "id BIGINT, name STRING")
start = time.time()
via_join = ids.join(lookup_df, "id", "left")
via_join.write.format("noop").mode("overwrite").save()
print(f"join        : {time.time() - start:.2f}s")

print("same names:", via_udf.orderBy("id").limit(5).collect() == via_join.select("id", "name").orderBy("id").limit(5).collect())

## 5. Python ↔ JVM: pickle vs Arrow

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Converts a 1-million-row DataFrame to pandas and back, and times a pickle-based UDF against an Arrow-based pandas UDF.<br><br>
<b>Why it matters</b><br>Arrow moves <b>columns</b> in batches using a shared binary format, while pickle serializes <b>values one at a time</b>. That's why <code>toPandas()</code> with Arrow and pandas UDFs are much faster than row-by-row conversion. Spark Connect always uses Arrow to return results to the notebook.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A <code>toPandas</code> time and a <code>createDataFrame(pandas)</code> time for 1 million rows, then the two UDF timings. For a trivial <code>+ 1</code>, the fixed cost of starting Python workers can make them close. The pandas UDF pulls ahead as the number of rows and the per-row work grow, because it avoids per-value pickling.
</div>

In [0]:
import pandas as pd
from pyspark.sql.functions import pandas_udf

df = spark.range(1_000_000).withColumn("v", F.rand(seed=1))

start = time.time(); pdf = df.toPandas(); print(f"toPandas (Arrow)        : {time.time() - start:.2f}s, rows={len(pdf):,}")
start = time.time(); back = spark.createDataFrame(pdf); back.count(); print(f"createDataFrame(pandas) : {time.time() - start:.2f}s")

@udf("double")
def plus_one_pickle(v):
    return v + 1

@pandas_udf("double")
def plus_one_arrow(v: pd.Series) -> pd.Series:
    return v + 1

for label, col in [("classic UDF (pickle)", plus_one_pickle("v")), ("pandas UDF (Arrow)", plus_one_arrow("v"))]:
    start = time.time()
    df.select(col.alias("x")).write.format("noop").mode("overwrite").save()
    print(f"{label:<24}: {time.time() - start:.2f}s")

## 6. Per-partition setup instead of capturing objects

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Uses <code>mapInPandas</code> to create an expensive object <b>once per partition</b> on the executor, instead of capturing it from the driver.<br><br>
<b>Why it matters</b><br>For resources such as clients, models or connections, the right pattern is to create them where they're used. <code>mapInPandas</code> gives you an iterator over each partition's batches, so setup code runs once per partition rather than once per row (or not at all, if it can't be pickled).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
4 rows, one per partition, each reporting how many rows that partition's single "client" processed.
</div>

In [0]:
def per_partition(batches):
    import threading
    client_lock = threading.Lock()          # created on the executor: no pickling needed
    processed = 0
    for batch in batches:
        with client_lock:
            processed += len(batch)
    yield pd.DataFrame({"rows_processed": [processed]})

spark.range(0, 1000, 1, numPartitions=4).mapInPandas(per_partition, "rows_processed LONG").show()

## Under the hood

```
DRIVER                                   EXECUTOR
task = (plan fragment + closures)
   │ serialize (Java / cloudpickle)
   └─────────────────────────────────▶   deserialize task
                                          JVM operators work on UnsafeRow (no object serialization)
                                          Python UDF: rows ──pickle/Arrow──▶ Python worker ──▶ back
                                          shuffle write: UnsafeRow bytes (+ compression)
```

<span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> serialization costs appear only when the job runs.

**Task size warnings:** "Stage contains a task of very large size" means closures captured large objects. Move the data into a DataFrame (join or broadcast) instead.

**Compression:** shuffle files and broadcast blocks are also compressed (`lz4` by default), trading a little CPU for much less disk and network I/O.

In [0]:
spark.range(10).select(times_multiplier("id")).explain()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>PicklingError: Could not serialize object</code></b><br>
The function captures something unpicklable (a lock, connection, SparkSession, client). Create it inside the function, or per partition with <code>mapInPandas</code>.<br><br>
<b>⛔ Problem: "task of very large size" warning, slow task launch</b><br>A big object is captured in a closure. Convert it to a DataFrame and join, or broadcast the DataFrame.<br><br>
**⛔ Problem: slow <code>toPandas()</code>**<br>Too much data, or Arrow disabled on classic compute. Aggregate first. Check <code>spark.sql.execution.arrow.pyspark.enabled</code>.<br><br>
<b>⛔ Problem: classic Python UDF is slow</b><br>Row-by-row pickling. Use built-ins, or a pandas UDF (Arrow).<br><br>
**⛔ Problem: <code>sparkContext.broadcast</code> not available**<br>Spark Connect (serverless) has no <code>sparkContext</code>. Use a DataFrame with a broadcast join.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. Where does serialization happen in Spark?</b><br>
When tasks and their closures are shipped to executors, when data is shuffled or cached, when data moves between the JVM and Python (UDFs, <code>toPandas</code>, <code>createDataFrame</code> from pandas), and when results are returned to the driver or client.<br><br>

<b>🎤 2. What is the difference between Java serialization and Kryo?</b><br>
Java serialization works for any <code>Serializable</code> class but is slow and produces large output. Kryo is faster and more compact, especially with registered classes. It mainly matters for RDDs of objects. DataFrames use Tungsten's binary format instead.<br><br>

<b>🎤 3. Why do DataFrames serialize more efficiently than RDDs?</b><br>
Because the schema is known, rows are stored in Tungsten's compact binary format and can be shuffled, cached, hashed and compared without generic object serialization.<br><br>

<b>🎤 4. Why is a Python UDF slow, from a serialization point of view?</b><br>
Every row is serialized from the JVM to a Python worker and the results are serialized back. Classic UDFs use pickle row by row. pandas and Arrow UDFs move columnar Arrow batches, which is much cheaper.<br><br>

<b>🎤 5. What causes <code>PicklingError: Could not serialize object</code>?</b><br>
A function sent to executors references an object that can't be pickled, such as a lock, a database connection, a file handle or the SparkSession. Create those objects inside the function or once per partition.<br><br>

<b>🎤 6. How do you share a large lookup table with executors?</b><br>
Put it in a DataFrame and join, letting Spark broadcast it if it's small. On classic compute, broadcast variables are another option, but they aren't available with Spark Connect.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A pandas UDF is usually faster than a regular Python UDF because:</b><br>
A. It runs on the driver<br>
B. It transfers data in columnar Apache Arrow batches and processes whole pandas Series<br>
C. It uses Kryo serialization<br>
D. It disables shuffles<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. A UDF that references a database connection object created in the notebook fails with a serialization error. What is the best fix?</b><br>
A. Increase executor memory<br>
B. Create the connection inside the function or once per partition (for example with <code>mapInPandas</code>)<br>
C. Cache the DataFrame first<br>
D. Switch to Kryo serialization<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Write a UDF that captures a list of 1 million numbers and returns whether <code>id</code> is in it. Then rewrite the logic with a DataFrame and a <code>left_semi</code> join. Compare the times</li><li>Trigger a serialization error on purpose by capturing an open file handle in a UDF, then fix it</li><li>Use <code>mapInPandas</code> to add a column with the partition's row count to every row</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Captured list vs semi join
allowed = set(range(0, 2_000_000, 2))                 # a set captured in the closure
@udf("boolean")
def is_allowed(x):
    return x in allowed

nums = spark.range(0, 200_000, 1, numPartitions=8)
start = time.time(); nums.filter(is_allowed("id")).write.format("noop").mode("overwrite").save()
print(f"closure UDF   : {time.time() - start:.2f}s")
allowed_df = spark.range(0, 2_000_000, 2).withColumnRenamed("id", "id")
start = time.time(); nums.join(allowed_df, "id", "left_semi").write.format("noop").mode("overwrite").save()
print(f"left_semi join: {time.time() - start:.2f}s")

# 2. Unpicklable file handle, then the fix
import tempfile, os
handle = open(os.path.join(tempfile.gettempdir(), "spark_demo.txt"), "w")
@udf("long")
def bad(x):
    handle.write("x"); return x
try:
    spark.range(2).select(bad("id")).collect()
except Exception as e:
    print("error:", type(e).__name__)
handle.close()

@udf("long")
def good(x):
    return x          # open any resources here, inside the function, if you really need them
print(spark.range(2).select(good("id")).collect())

# 3. Partition row count on every row
def add_count(batches):
    frames = list(batches)
    total = sum(len(f) for f in frames)
    for f in frames:
        yield f.assign(partition_rows=total)

spark.range(0, 10, 1, numPartitions=3).mapInPandas(add_count, "id LONG, partition_rows LONG").show()

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Spark serializes closures (to executors), shuffle and cache data, and data crossing JVM ↔ Python</li><li>DataFrames use Tungsten binary rows. RDDs use Java or Kryo serialization. Kryo is faster for objects</li><li>Python: classic UDFs pickle per row, pandas/Arrow UDFs and <code>toPandas</code> use Arrow batches</li><li>Captured objects travel with every task. Big ones slow tasks, unpicklable ones fail</li><li>Use DataFrame joins (broadcast) for lookups, and create resources per partition with <code>mapInPandas</code></li><li><code>spark.serializer</code> is a cluster setting, not changeable on serverless</li></ul>
</div>

| Problem | Fix |
|---|---|
| `PicklingError` | Create the object inside the function / per partition |
| Big lookup in a UDF | DataFrame + broadcast join |
| Slow UDF | Built-ins, or `@pandas_udf` |
| Slow `toPandas` | Aggregate first. Arrow enabled |
| Per-partition setup | `df.mapInPandas(func, schema)` |

## Git commit

```
git add 02_spark_internals/11_serialization.ipynb
git commit -m "add 02_11 serialization notebook"
```